In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
from shapely.affinity import translate
from shapely import vectorized

# =========================
# 0. 参数设置
# =========================

RA_MIN, RA_MAX = 0, 10
DEC_MIN, DEC_MAX = 0, 10

DENSITY = 8000  # /deg^2
AREA = (RA_MAX - RA_MIN) * (DEC_MAX - DEC_MIN)
N_TARGET = int(DENSITY * AREA)

N_TILES = 50
N_FIBER = 200   # 每个 module fiber 数
N0 = 2000       # 控制 fill fraction

N_STEPS = 200   # 优化步数

# =========================
# 1. 生成 target
# =========================

ra = np.random.uniform(RA_MIN, RA_MAX, N_TARGET)
dec = np.random.uniform(DEC_MIN, DEC_MAX, N_TARGET)
targets = np.vstack([ra, dec]).T

print(f"Generated {len(targets)} targets")

# =========================
# 2. 读取 PLY
# =========================

def read_ply(filename):
    vertices = []
    faces = []

    with open(filename, 'r') as f:
        assert f.readline().strip() == 'ply'

        num_vertices = 0
        num_faces = 0

        while True:
            line = f.readline().strip()
            if line.startswith('element vertex'):
                num_vertices = int(line.split()[-1])
            elif line.startswith('element face'):
                num_faces = int(line.split()[-1])
            elif line == 'end_header':
                break

        for _ in range(num_vertices):
            x, y, z = map(float, f.readline().split())
            vertices.append([x, y, z])

        for _ in range(num_faces):
            parts = list(map(int, f.readline().split()))
            faces.append(parts[1:])

    return np.array(vertices), faces


# =========================
# 3. mesh → modules
# =========================

def mesh_to_modules(vertices, faces, scale=1.0):
    modules = []

    for face in faces:
        pts = vertices[face][:, :2] * scale
        poly = Polygon(pts)

        if poly.is_valid and poly.area > 0:
            modules.append(poly)

    return modules


# =========================
# 4. fiber fill fraction
# =========================

def fiber_fill_fraction(N):
    """
    你的核心模型
    """
    return 1 - np.exp(-N / N0)


def observe_in_module(idx, N_fiber):
    N_target = len(idx)

    if N_target == 0:
        return np.array([], dtype=int)

    f = fiber_fill_fraction(N_target)

    N_obs = int(f * N_fiber)
    N_obs = min(N_obs, N_target)

    if N_obs == 0:
        return np.array([], dtype=int)

    return np.random.choice(idx, N_obs, replace=False)


# =========================
# 5. tile观测
# =========================

def observe_in_tile(tile_center, targets, modules):
    observed = set()

    x = targets[:, 0]
    y = targets[:, 1]

    for module in modules:
        shifted = translate(module, xoff=tile_center[0], yoff=tile_center[1])

        mask = vectorized.contains(shifted, x, y)
        idx = np.where(mask)[0]

        obs_idx = observe_in_module(idx, N_FIBER)
        observed.update(obs_idx)

    return observed


# =========================
# 6. evaluate
# =========================

def evaluate(tiles, targets, modules):
    observed = set()

    for tile in tiles:
        obs = observe_in_tile(tile, targets, modules)
        observed.update(obs)

    return len(observed)


# =========================
# 7. 初始化 tiles
# =========================

tiles = np.vstack([
    np.random.uniform(RA_MIN, RA_MAX, N_TILES),
    np.random.uniform(DEC_MIN, DEC_MAX, N_TILES)
]).T


# =========================
# 8. 加载 focal plane
# =========================

vertices, faces = read_ply("/home/hmf/work/FA_test/data/focalplane_ply/2025-10-27T17:01:02+00:00_module_edges.ply")

# ⚠️ 如果单位是 mm，需要调这个！
SCALE = 1.0   # 例如：1/3600

modules = mesh_to_modules(vertices, faces, scale=SCALE)

print(f"Loaded {len(modules)} modules")


# =========================
# 9. 优化（greedy）
# =========================

best_tiles = tiles.copy()
best_score = evaluate(best_tiles, targets, modules)

print("Initial score:", best_score)

for step in range(N_STEPS):

    trial_tiles = best_tiles.copy()

    i = np.random.randint(0, N_TILES)

    trial_tiles[i, 0] += np.random.normal(0, 0.3)
    trial_tiles[i, 1] += np.random.normal(0, 0.3)

    trial_tiles[i, 0] = np.clip(trial_tiles[i, 0], RA_MIN, RA_MAX)
    trial_tiles[i, 1] = np.clip(trial_tiles[i, 1], DEC_MIN, DEC_MAX)

    score = evaluate(trial_tiles, targets, modules)

    if score > best_score:
        best_score = score
        best_tiles = trial_tiles
        print(f"Step {step}: improved -> {best_score}")


# =========================
# 10. 可视化
# =========================

plt.figure(figsize=(8, 8))

plt.scatter(targets[:, 0], targets[:, 1], s=1, alpha=0.2)

for tile in best_tiles:
    for module in modules:
        shifted = translate(module, xoff=tile[0], yoff=tile[1])
        x, y = shifted.exterior.xy
        plt.plot(x, y, 'r-', alpha=0.2)

plt.xlim(RA_MIN, RA_MAX)
plt.ylim(DEC_MIN, DEC_MAX)
plt.xlabel("RA")
plt.ylabel("DEC")
plt.title(f"Observed targets = {best_score}")

plt.show()

Generated 800000 targets
Loaded 336 modules


/tmp/ipykernel_2106833/176558278.py:126: DeprecationWarning: The 'shapely.vectorized.contains' function is deprecated and will be removed a future version. Use 'shapely.contains_xy' instead (available since shapely 2.0.0).
  mask = vectorized.contains(shifted, x, y)


Initial score: 41
